# Trabalho disciplina LLM & DL

**Dataset**: [ai-lab/MBD-mini](https://huggingface.co/datasets/ai-lab/MBD-mini) — Multimodal Banking Dataset (subset 10%)

**Objetivo**: Prever a propensão de clientes adquirir 4 produtos bancários no mês seguinte.

**Métrica**: ROC-AUC por target

**Aluno**: Iago Garcia Vargas

## 1. Setup e Download dos Dados

In [ ]:
!pip install -q torch scikit-learn xgboost lightgbm pandas numpy matplotlib seaborn huggingface-hub pyarrow tqdm

import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE_DIR = "/content/drive/MyDrive/transformers_financial_v4"
    os.makedirs(SAVE_DIR, exist_ok=True)
    print(f"Artifacts will be saved to: {SAVE_DIR}")
    ON_COLAB = True
except ImportError:
    SAVE_DIR = "transformers_financial_v4"
    os.makedirs(SAVE_DIR, exist_ok=True)
    print(f"Not on Colab. Artifacts will be saved locally to: {SAVE_DIR}")
    ON_COLAB = False

In [ ]:
import os
import tarfile
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from huggingface_hub import hf_hub_download
from scipy.stats import entropy
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import QuantileTransformer
from sklearn.decomposition import PCA
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from tqdm.auto import tqdm
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

In [ ]:
REPO_ID = "ai-lab/MBD-mini"
FILES = ["detail.tar.gz", "targets.tar.gz", "client_split.tar.gz"]

for fname in FILES:
    archive = DATA_DIR / fname
    extract_dir = DATA_DIR / fname.replace(".tar.gz", "")

    # 1. Download
    if not archive.exists():
        print(f"Downloading {fname}...")
        hf_hub_download(
            repo_id=REPO_ID,
            filename=fname,
            repo_type="dataset",
            local_dir=str(DATA_DIR),
        )

    # 2. Extração
    if not extract_dir.exists():
        print(f"Extracting {fname}...")
        with tarfile.open(archive, "r:gz") as tar:
            tar.extractall(path=DATA_DIR)

In [ ]:
all_parquet = list(DATA_DIR.rglob("*.parquet"))

trx_files = sorted([f for f in all_parquet if "detail/trx" in f.as_posix()])
target_files = sorted([f for f in all_parquet if "targets" in f.as_posix() and "detail" not in f.as_posix()])
split_files = sorted([f for f in all_parquet if "client_split" in f.as_posix()])

print(f"Files found - Trx: {len(trx_files)} | Target: {len(target_files)} | Split: {len(split_files)}")

datasets = {}
for name, files in [("trx", trx_files), ("targets", target_files), ("splits", split_files)]:
    if not files:
        print(f"Warning: No files found for {name}!")
        datasets[name] = pd.DataFrame()
        continue

    datasets[name] = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)
    print(f"{name.capitalize()} shape: {datasets[name].shape}")

df_trx, df_targets, df_splits = datasets["trx"], datasets.get("targets"), datasets.get("splits")

display(df_trx.head())

## 2. Análise Exploratória (EDA)

In [ ]:
TARGET_COLS = ["target_1", "target_2", "target_3", "target_4"]
total_rows = len(df_targets)

print(
    f"=== Targets Overview ===\n"
    f"Unique clients: {df_targets['client_id'].nunique():,}\n"
    f"Months ({df_targets['mon'].nunique()}): {sorted(df_targets['mon'].unique())}\n\n"
    f"=== Target Distribution ==="
)

for col in TARGET_COLS:
    pos = df_targets[col].sum()
    # Usando formatação nativa de porcentagem do Python
    print(f"{col}: {pos:,} / {total_rows:,} ({pos / total_rows:.2%})")

display(df_targets.head())

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, col in zip(axes, TARGET_COLS):
    counts = df_targets[col].value_counts().sort_index()

    bars = ax.bar(
        counts.index.astype(str),
        counts.values,
        color=["#4C72B0", "#DD8452"][:len(counts)]
    )

    ax.set_title(col)
    ax.spines[["top", "right"]].set_visible(False)

    ax.bar_label(bars, fmt='{:,}', padding=3, fontsize=9)

axes[0].set_ylabel("Count")
fig.suptitle("Target Distribution", fontsize=14, fontweight="bold", y=1.05)

plt.tight_layout()
plt.show()

In [ ]:
print(
    f"=== Transaction Data Overview ===\n"
    f"Total transactions: {len(df_trx):,}\n"
    f"Unique clients:     {df_trx['client_id'].nunique():,}\n"
)
display(df_trx.dtypes.to_frame("Data Type"))

trx_per_client = df_trx.groupby("client_id").size()
print("\nTransactions per client:")
display(trx_per_client.describe().to_frame("Metrics"))

fig, (ax_trx, ax_amt, ax_evt) = plt.subplots(1, 3, figsize=(18, 4))

ax_trx.hist(trx_per_client.clip(upper=trx_per_client.quantile(0.99)), bins=50, color="#4C72B0", edgecolor="white")
ax_trx.set(title="Transactions per Client (99th pct)", xlabel="Number of transactions", ylabel="Clients")

q01, q99 = df_trx["amount"].quantile([0.01, 0.99])
ax_amt.hist(df_trx["amount"].clip(lower=q01, upper=q99), bins=50, color="#4C72B0", edgecolor="white")
ax_amt.set(title="Transaction Amount (1st-99th pct)", xlabel="Amount")

top_events = df_trx["event_type"].value_counts().nlargest(20)
ax_evt.barh(top_events.index.astype(str), top_events.values, color="#DD8452")
ax_evt.set(title="Top 20 Event Types", xlabel="Count")
ax_evt.invert_yaxis()

for ax in (ax_trx, ax_amt, ax_evt):
    ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
if "event_time" in df_trx.columns:
    if not pd.api.types.is_datetime64_any_dtype(df_trx["event_time"]):
        df_trx["event_time"] = pd.to_datetime(df_trx["event_time"])

    daily_counts = df_trx.groupby(df_trx["event_time"].dt.floor("d")).size()

    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(daily_counts.index, daily_counts.values, linewidth=1.5, color="#4C72B0")

    ax.set(title="Daily Transaction Volume Over Time", xlabel="Date", ylabel="Transactions")
    ax.spines[["top", "right"]].set_visible(False)

    plt.tight_layout()
    plt.show()

ignore_cols = {"client_id", "event_time", "amount", "fold"}
CAT_COLS = [c for c in df_trx.columns if c not in ignore_cols]

print(f"Categorical columns ({len(CAT_COLS)}): {CAT_COLS}\n")
print("Cardinality:")

cardinality = df_trx[CAT_COLS].nunique().to_frame("Unique Values")
display(cardinality)

## 3. Preprocessamento

- **Sliding Windows**: cada par (cliente, mês) gera um exemplo transações até mês T predizem targets do mês T + 1
- **Log1p + QuantileTransformer** no amount para lidar com distribuição heavy-tail
- **Embeddings cíclicos**: extrair day_of_week e day_of_month de event_time
- Folds 0-2 para treino, fold 3 para validação, fold 4 para teste

In [ ]:
df_splits = pd.concat(
    [pd.read_parquet(f).assign(fold=int(f.parent.name.split("=")[1])) for f in split_files],
    ignore_index=True
)

mon_min, mon_max = df_targets["mon"].min(), df_targets["mon"].max()
print(f"Available months: {df_targets['mon'].nunique()} ({mon_min} → {mon_max})")

labels_sw = df_targets.merge(df_splits, on="client_id", how="inner")
n_sw = len(labels_sw)
n_clients = labels_sw["client_id"].nunique()

print(f"\nSliding window examples: {n_sw:,} (vs {n_clients:,} unique clients)")
print(f"Average windows per client: {n_sw / n_clients:.1f}")

labels_last = (
    df_targets.sort_values("mon")
    .drop_duplicates(subset=["client_id"], keep="last")
    .merge(df_splits, on="client_id", how="inner")
)
labels = labels_last

sw_train = labels_sw[labels_sw["fold"] < 3].copy()
sw_val = labels_sw[labels_sw["fold"] == 3].copy()
sw_test_last = labels_last[labels_last["fold"] == 4].copy()

print(f"\nTrain clients: {sw_train['client_id'].nunique():,} ({len(sw_train):,} sliding window examples)")
print(f"Val clients:   {sw_val['client_id'].nunique():,} ({len(sw_val):,} sliding window examples)")
print(f"Test clients:  {sw_test_last['client_id'].nunique():,} ({len(sw_test_last):,} examples, last month only)")

In [ ]:
train_clients = set(sw_train["client_id"])
val_clients = set(sw_val["client_id"])
test_clients = set(sw_test_last["client_id"])

valid_clients = train_clients | val_clients | test_clients

df_trx = (
    df_trx[df_trx["client_id"].isin(valid_clients)]
    .sort_values(["client_id", "event_time"])
    .reset_index(drop=True)
)
print(f"Transactions after filtering: {len(df_trx):,}")

train_mask = df_trx["client_id"].isin(train_clients)

CAT_COLS = [c for c in df_trx.columns if c not in {"client_id", "event_time", "amount", "fold"}]
cat_mappings = {}

for col in CAT_COLS:
    unique_vals = np.sort(df_trx.loc[train_mask, col].dropna().unique())
    mapping = {v: i + 1 for i, v in enumerate(unique_vals)}
    cat_mappings[col] = mapping

    df_trx[col] = df_trx[col].map(mapping).fillna(0).astype(np.int32)

CAT_CARDS = {col: len(mapping) + 1 for col, mapping in cat_mappings.items()}
print("\nCategorical cardinalities (including padding idx 0):")
display(pd.Series(CAT_CARDS, name="Cardinality").to_frame())

amount_arr = df_trx["amount"].values
amount_log = np.sign(amount_arr) * np.log1p(np.abs(amount_arr))

qt = QuantileTransformer(n_quantiles=1000, output_distribution="normal", random_state=SEED)
qt.fit(amount_log[train_mask].reshape(-1, 1))

df_trx["amount_scaled"] = qt.transform(amount_log.reshape(-1, 1)).astype(np.float32)

print(f"\nAmount transform: raw → log1p → QuantileTransformer(normal)")
print(f"  Before: mean={amount_arr.mean():.2f}, std={amount_arr.std():.2f}, max={amount_arr.max():.2f}")
print(f"  After:  mean={df_trx['amount_scaled'].mean():.4f}, std={df_trx['amount_scaled'].std():.4f}")

event_time = df_trx["event_time"]
ref_time = event_time[train_mask].min()

df_trx["time_delta_days"] = ((event_time - ref_time).dt.total_seconds() / 86400.0).astype(np.float32)

dow = event_time.dt.dayofweek
dom = event_time.dt.day - 1

df_trx["dow_sin"] = np.sin(2 * np.pi * dow / 7).astype(np.float32)
df_trx["dow_cos"] = np.cos(2 * np.pi * dow / 7).astype(np.float32)
df_trx["dom_sin"] = np.sin(2 * np.pi * dom / 31).astype(np.float32)
df_trx["dom_cos"] = np.cos(2 * np.pi * dom / 31).astype(np.float32)

df_trx["trx_month"] = event_time.dt.to_period("M")

CONT_COLS = ["amount_scaled", "dow_sin", "dow_cos", "dom_sin", "dom_cos"]
print(f"\nContinuous features ({len(CONT_COLS)}): {CONT_COLS}")

display(df_trx[["client_id", "amount", "amount_scaled", "dow_sin", "dow_cos", "dom_sin", "dom_cos", "time_delta_days"]].head())

## 4. Baselines Tradicionais

### 4.1 Feature Engineering Agregada

Abordagem clássica: agregar sequências de transações por cliente em vetores de features estatísticas.

Obs: não precisa executar, o df processado já está salvo (demora muito executar do zero)

In [ ]:
from scipy.stats import entropy

def build_aggregated_features(df_trx: pd.DataFrame, max_month=None) -> pd.DataFrame:
    """Build aggregated features per client_id."""
    df = df_trx[df_trx["trx_month"] <= max_month] if max_month is not None else df_trx

    aggs = {
        "amount_mean": ("amount", "mean"),
        "amount_std": ("amount", "std"),
        "amount_min": ("amount", "min"),
        "amount_max": ("amount", "max"),
        "amount_sum": ("amount", "sum"),
        "amount_median": ("amount", "median"),
        "trx_count": ("amount", "count"),
    }

    if "time_delta_days" in df.columns:
        aggs["time_first"] = ("time_delta_days", "min")
        aggs["time_last"] = ("time_delta_days", "max")

    for col in CAT_COLS:
        aggs[f"{col}_nunique"] = (col, "nunique")
        aggs[f"{col}_mode"] = (col, lambda x: x.value_counts().index[0] if len(x) else 0)
        aggs[f"{col}_entropy"] = (col, lambda x: entropy(np.unique(x, return_counts=True)[1]))

    features = df.groupby("client_id").agg(**aggs)
    features["amount_std"] = features["amount_std"].fillna(0)

    if "time_delta_days" in df.columns:
        features["time_span"] = features["time_last"] - features["time_first"]

        features["avg_days_between_trx"] = np.where(
            features["trx_count"] > 1,
            features["time_span"] / (features["trx_count"] - 1),
            0
        )

    return features.fillna(0)


df_features = build_aggregated_features(df_trx)
print(f"Aggregated features shape: {df_features.shape}")
print(f"Features: {list(df_features.columns)[:15]}...\n")

print("Pre-computing per-month aggregated features for hybrid pipeline...")
all_months_for_agg = sorted(df_trx["trx_month"].unique()) if "trx_month" in df_trx.columns else []

df_features_by_month = {}
for mon in tqdm(all_months_for_agg, desc="Aggregating per month"):
    df_features_by_month[mon] = build_aggregated_features(df_trx, max_month=mon)

print(f"Computed features for {len(df_features_by_month)} months")
display(df_features.head())

In [ ]:
import pickle
from pathlib import Path

save_dir = Path(SAVE_DIR)

global_path = save_dir / "df_features_global.parquet"
monthly_path = save_dir / "df_features_by_month.pkl"

print("Saving aggregated features to disk...")
df_features.to_parquet(global_path)

with open(monthly_path, "wb") as f:
    pickle.dump(df_features_by_month, f)

print(f"Success! Files saved at:\n- {global_path}\n- {monthly_path}")

In [ ]:
import pickle
import pandas as pd

print("Loading aggregated features...")

save_dir = Path(SAVE_DIR)
global_path = save_dir / "df_features_global.parquet"
monthly_path = save_dir / "df_features_by_month.pkl"

if global_path.exists() and monthly_path.exists():
    df_features = pd.read_parquet(global_path)

    with open(monthly_path, "rb") as f:
        df_features_by_month = pickle.load(f)

    print("Features loaded successfully!")
    print(f"Global features shape: {df_features.shape}")
    print(f"Months loaded: {len(df_features_by_month)}")
else:
    print("Warning: Feature files not found. You need to process them first.")

In [ ]:
df_features.head()

In [ ]:
labels_idx = labels.set_index("client_id")

df_features, labels_aligned = df_features.align(labels_idx, join="inner", axis=0)

train_mask = labels_aligned["fold"] < 3
test_mask = labels_aligned["fold"] == 4

X_train = df_features[train_mask].values
X_test = df_features[test_mask].values

y_train = labels_aligned.loc[train_mask, TARGET_COLS].values
y_test = labels_aligned.loc[test_mask, TARGET_COLS].values

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"X_train: {X_train.shape} | y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape}  | y_test:  {y_test.shape}")

### 4.2 Treinamento dos Baselines

In [ ]:
def train_and_evaluate(model_class, model_name: str, X_train, y_train, X_test, y_test, **kwargs) -> dict:
    results = {}
    aucs = []

    for i, target in enumerate(TARGET_COLS):
        model = model_class(**kwargs)
        model.fit(X_train, y_train[:, i])

        if hasattr(model, "predict_proba"):
            y_pred = model.predict_proba(X_test)[:, 1]
        else:
            y_pred = model.predict(X_test)

        try:
            auc = roc_auc_score(y_test[:, i], y_pred)
        except ValueError:
            auc = 0.5

        aucs.append(auc)
        results[target] = auc

    results["mean_auc"] = np.mean(aucs)

    print(f"[{model_name}]")
    for target in TARGET_COLS:
        print(f"  {target}: {results[target]:.4f}")
    print(f"  Mean AUC: {results['mean_auc']:.4f}\n")

    return results

models_config = {
    "Logistic Regression": (LogisticRegression, {
        "max_iter": 1000, "random_state": SEED, "class_weight": "balanced"
    }),
    "Random Forest": (RandomForestClassifier, {
        "n_estimators": 200, "max_depth": 10, "random_state": SEED,
        "class_weight": "balanced", "n_jobs": -1
    }),
    "XGBoost": (XGBClassifier, {
        "n_estimators": 300, "max_depth": 6, "learning_rate": 0.1,
        "random_state": SEED, "eval_metric": "logloss",
        "verbosity": 0, "scale_pos_weight": 10
    }),
    "LightGBM": (LGBMClassifier, {
        "n_estimators": 300, "max_depth": 6, "learning_rate": 0.1,
        "random_state": SEED, "is_unbalance": True, "verbose": -1
    })
}

all_results = {}
for name, (cls, params) in models_config.items():
    all_results[name] = train_and_evaluate(
        cls, name, X_train, y_train, X_test, y_test, **params
    )

In [ ]:
baseline_df = pd.DataFrame(all_results).T
baseline_df.columns = TARGET_COLS + ["Mean AUC"]
baseline_df = baseline_df.round(4)
print("=== Baseline Results ===")
baseline_df

## 5. Arquitetura do Transformer

Arquitetura modular com **encoder compartilhado** para pré-treinamento e classificação:

1. **Encoder**: Embeddings categóricos + **5 features contínuas** (amount + 4 cyclic) → projeção linear → Temporal PE sinusoidal → Transformer Encoder (Pre-LN) → CLS token
2. **MTM Head** (pré-treinamento): reconstrói features mascaradas, usando cross-entropy para categóricas, MSE para amount + cyclic
3. **Classification Head** (fine-tuning): CLS token → MLP → logits para 4 targets

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

class TemporalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_period=10000.0):
        super().__init__()
        self.d_model = d_model
        self.max_period = max_period

    def forward(self, time_deltas):
        half_d = self.d_model // 2
        freqs = torch.exp(
            torch.arange(half_d, device=time_deltas.device, dtype=torch.float32)
            * -(np.log(self.max_period) / half_d)
        )
        args = time_deltas.unsqueeze(-1) * freqs
        pe = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)

        # F.pad é mais idiomático e limpo que concatenar zeros
        if self.d_model % 2 != 0:
            pe = F.pad(pe, (0, 1))
        return pe

class TransactionTransformer(nn.Module):
    def __init__(self, cat_cardinalities, cat_cols, d_model=128, nhead=4,
                 num_layers=3, dim_feedforward=256, dropout=0.1,
                 num_targets=4, emb_dim=16, n_cont_features=5):
        super().__init__()
        self.cat_cols = cat_cols
        self.d_model = d_model

        # --- Encoder ---
        self.cat_embeddings = nn.ModuleDict({
            col: nn.Embedding(card, emb_dim, padding_idx=0)
            for col, card in cat_cardinalities.items()
        })

        self.input_proj = nn.Sequential(
            nn.Linear((emb_dim * len(cat_cols)) + n_cont_features, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.temporal_pe = TemporalPositionalEncoding(d_model)
        self.input_norm = nn.LayerNorm(d_model)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)

        self.transformer_encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
                dropout=dropout, batch_first=True, activation="gelu", norm_first=True,
            ),
            num_layers=num_layers
        )
        self.out_norm = nn.LayerNorm(d_model)

        # --- MTM Head (pre-training) ---
        self.mtm_norm = nn.LayerNorm(d_model)
        self.mtm_cat_heads = nn.ModuleDict({
            col: nn.Linear(d_model, card) for col, card in cat_cardinalities.items()
        })
        self.mtm_cont_head = nn.Linear(d_model, n_cont_features)

        # --- Classification Head (fine-tuning) ---
        self.classifier = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(d_model // 2, num_targets),
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def _prepare_sequence(self, cat_features, cont_features, time_deltas, padding_mask):
        batch_size = cont_features.size(0)

        cat_embs = [self.cat_embeddings[col](cat_features[col]) for col in self.cat_cols]
        x = torch.cat(cat_embs + [cont_features], dim=-1)

        x = self.input_proj(x) + self.temporal_pe(time_deltas)
        x = self.input_norm(x)

        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)

        cls_pad = torch.zeros(batch_size, 1, dtype=torch.bool, device=x.device)
        extended_mask = torch.cat([cls_pad, padding_mask], dim=1)

        return x, extended_mask

    def encode(self, cat_features, cont_features, time_deltas, padding_mask):
        x, extended_mask = self._prepare_sequence(cat_features, cont_features, time_deltas, padding_mask)
        x = self.transformer_encoder(x, src_key_padding_mask=extended_mask)
        return self.out_norm(x[:, 0]), x[:, 1:]

    def forward_pretrain(self, cat_features, cont_features, time_deltas, padding_mask, mask_positions):
        _, seq_out = self.encode(cat_features, cont_features, time_deltas, padding_mask)
        masked_hidden = self.mtm_norm(seq_out[mask_positions])

        cat_logits = {col: head(masked_hidden) for col, head in self.mtm_cat_heads.items()}
        return cat_logits, self.mtm_cont_head(masked_hidden)

    def forward(self, cat_features, cont_features, time_deltas, padding_mask,
                return_embedding=False, return_attention=False):
        if return_attention:
            return self._forward_with_attention(cat_features, cont_features, time_deltas, padding_mask)

        cls_out, _ = self.encode(cat_features, cont_features, time_deltas, padding_mask)

        if return_embedding:
            return cls_out
        return self.classifier(cls_out)

    def _forward_with_attention(self, cat_features, cont_features, time_deltas, padding_mask):
        x, extended_mask = self._prepare_sequence(cat_features, cont_features, time_deltas, padding_mask)
        attn_weights_all = []

        for layer in self.transformer_encoder.layers:
            x_new = layer(x, src_key_padding_mask=extended_mask)

            with torch.no_grad():
                nhead = layer.self_attn.num_heads
                d_k = self.d_model // nhead
                bsz, sq = x.size(0), x.size(1)

                W_q, W_k, _ = layer.self_attn.in_proj_weight.chunk(3, dim=0)

                q = (x @ W_q.T).view(bsz, sq, nhead, d_k).transpose(1, 2)
                k = (x @ W_k.T).view(bsz, sq, nhead, d_k).transpose(1, 2)

                attn = (q @ k.transpose(-2, -1)) / (d_k ** 0.5)
                attn = attn.masked_fill(extended_mask.unsqueeze(1).unsqueeze(2), float('-inf'))
                attn_weights_all.append(torch.softmax(attn, dim=-1).mean(dim=1))

            x = x_new

        cls_output = self.out_norm(x[:, 0])
        return self.classifier(cls_output), attn_weights_all


ENCODER_PREFIXES = (
    'cat_embeddings.', 'input_proj.', 'temporal_pe.', 'input_norm.',
    'cls_token', 'transformer_encoder.', 'out_norm.',
)

def get_encoder_state(model):
    return {k: v.clone() for k, v in model.state_dict().items()
            if k.startswith(ENCODER_PREFIXES)}

def load_encoder_state(model, encoder_state):
    model.load_state_dict(encoder_state, strict=False)
    return model

## 6. Datasets e DataLoaders

Dataset com **Sliding Windows**:
- Cada exemplo é um par (client_id, month) com transações até aquele mês
- Pré-treinamento: apenas train_clients (pra evitar leakage)
- Classificação: sliding windows para train/val, last-month para test
- Continuous features: amount_scaled + 4 cyclic (dow_sin, dow_cos, dom_sin, dom_cos)

In [ ]:
MAX_SEQ_LEN = 512

class SlidingWindowDataset(Dataset):
    def __init__(self, labels_df, df_trx, cat_cols, cont_cols, max_seq_len=MAX_SEQ_LEN):
        self.examples = []
        grouped_trx = df_trx.groupby("client_id")

        for row in tqdm(labels_df.itertuples(index=False), total=len(labels_df), desc="Building SW dataset"):
            cid = row.client_id

            if cid not in grouped_trx.groups:
                continue

            client_trx = grouped_trx.get_group(cid)

            if "trx_month" in client_trx.columns:
                target_period = row.mon if isinstance(row.mon, pd.Period) else pd.Period(row.mon, freq="M")
                client_trx = client_trx[client_trx["trx_month"] <= target_period]

            client_trx = client_trx.tail(max_seq_len)

            if client_trx.empty:
                continue

            targets = np.array([getattr(row, tc) for tc in TARGET_COLS], dtype=np.float32)

            self.examples.append({
                "client_id": cid,
                "target_month": row.mon if isinstance(row.mon, pd.Period) else pd.Period(row.mon, freq="M"),
                "cat": {col: client_trx[col].values.astype(np.int64) for col in cat_cols},
                "cont": client_trx[cont_cols].values.astype(np.float32),
                "time": client_trx["time_delta_days"].values.astype(np.float32),
                "targets": targets,
                "length": len(client_trx),
            })

    def __len__(self): return len(self.examples)
    def __getitem__(self, idx): return self.examples[idx]


class PreTrainDataset(Dataset):
    def __init__(self, client_ids, df_trx, cat_cols, cont_cols, max_seq_len=MAX_SEQ_LEN):
        self.examples = []
        grouped = df_trx.groupby("client_id")
        rng = np.random.RandomState(SEED)

        all_months = df_trx["trx_month"].unique() if "trx_month" in df_trx.columns else []

        for cid in tqdm(client_ids, desc="Building Pretrain dataset"):
            if cid not in grouped.groups:
                continue

            client_trx = grouped.get_group(cid)

            if len(all_months) > 0 and "trx_month" in client_trx.columns:
                cutoff = rng.choice(all_months)
                client_trx = client_trx[client_trx["trx_month"] <= cutoff]

            client_trx = client_trx.tail(max_seq_len)

            if client_trx.empty:
                continue

            self.examples.append({
                "client_id": cid,
                "cat": {col: client_trx[col].values.astype(np.int64) for col in cat_cols},
                "cont": client_trx[cont_cols].values.astype(np.float32),
                "time": client_trx["time_delta_days"].values.astype(np.float32),
                "length": len(client_trx),
            })

    def __len__(self): return len(self.examples)
    def __getitem__(self, idx): return self.examples[idx]


def collate_fn(batch):
    """Agrupa batch pre-alocando tensores para máxima performance na GPU."""
    B = len(batch)
    max_len = max(item["length"] for item in batch)
    has_labels = "targets" in batch[0]

    cat_features = {col: torch.zeros(B, max_len, dtype=torch.long) for col in batch[0]["cat"]}

    n_cont = batch[0]["cont"].shape[-1]
    cont_features = torch.zeros(B, max_len, n_cont)

    time_deltas = torch.zeros(B, max_len)
    padding_mask = torch.ones(B, max_len, dtype=torch.bool)

    targets = torch.zeros(B, len(TARGET_COLS)) if has_labels else None

    for i, item in enumerate(batch):
        L = item["length"]

        for col in cat_features:
            cat_features[col][i, :L] = torch.from_numpy(item["cat"][col])

        cont_features[i, :L] = torch.from_numpy(item["cont"].reshape(L, n_cont))
        time_deltas[i, :L] = torch.from_numpy(item["time"])

        padding_mask[i, :L] = False

        if has_labels:
            targets[i] = torch.from_numpy(item["targets"])

    if has_labels:
        return cat_features, cont_features, time_deltas, padding_mask, targets
    return cat_features, cont_features, time_deltas, padding_mask

In [ ]:
print("Building PRE-TRAINING dataset (train clients only)...")
pretrain_dataset = PreTrainDataset(
    client_ids=list(train_clients), df_trx=df_trx, cat_cols=CAT_COLS,
    cont_cols=CONT_COLS, max_seq_len=MAX_SEQ_LEN
)

print("Building SLIDING WINDOW datasets...")
sw_kwargs = {
    "df_trx": df_trx,
    "cat_cols": CAT_COLS,
    "cont_cols": CONT_COLS,
    "max_seq_len": MAX_SEQ_LEN
}

train_dataset = SlidingWindowDataset(labels_df=sw_train, **sw_kwargs)
val_dataset = SlidingWindowDataset(labels_df=sw_val, **sw_kwargs)
test_dataset = SlidingWindowDataset(labels_df=sw_test_last, **sw_kwargs)

print(f"Sizes -> Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}\n")


BATCH_SIZE = 128
OVERSAMPLE_FACTOR = 10.0

is_positive = [ex["targets"].sum() > 0 for ex in train_dataset.examples]
weights = [OVERSAMPLE_FACTOR if pos else 1.0 for pos in is_positive]

train_sampler = WeightedRandomSampler(
    weights=torch.tensor(weights, dtype=torch.float64),
    num_samples=len(train_dataset),
    replacement=True
)


loader_kwargs = {"batch_size": BATCH_SIZE, "collate_fn": collate_fn, "num_workers": 0}

pretrain_loader = DataLoader(pretrain_dataset, shuffle=True, drop_last=True, **loader_kwargs)
train_loader = DataLoader(train_dataset, sampler=train_sampler, **loader_kwargs)
val_loader = DataLoader(val_dataset, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_kwargs)


n_pos = sum(is_positive)
pct_pos = (n_pos / len(train_dataset)) * 100

print(
    f"[Sampler] {n_pos:,} positive examples ({pct_pos:.1f}%) weighted {OVERSAMPLE_FACTOR}x\n"
    f"Pre-train batches: {len(pretrain_loader):,} | Train batches: {len(train_loader):,}"
)

## 7. Pré-treinamento: Masked Transaction Modeling (MTM)

Análogo ao Masked Language Modeling (MLM) do BERT:

1. **Mascarar 15%** das transações em cada sequência
2. Das posições mascaradas: 80% → zeros (token [MASK]), 10% → valores aleatórios, 10% → manter original
3. O modelo aprende a **reconstruir** as transações originais a partir do contexto sequencial
4. Loss = Cross-Entropy (features categóricas) + MSE (amount)
5. **Nenhum label** é usado, aprendizado auto-supervisionado

Isso força o Transformer a aprender padrões como sazonalidade, sequências típicas de transações, e correlações temporais.

In [ ]:
def mask_transactions(cat_feats, cont_feats, padding_mask, mask_prob=0.15, cat_cards=None):
    """Apply BERT-style masking to transaction features."""
    B, S = padding_mask.shape
    device = padding_mask.device
    valid = ~padding_mask

    mask = (torch.rand(B, S, device=device) < mask_prob) & valid

    invalid_mask = valid.any(dim=1) & ~mask.any(dim=1)
    if invalid_mask.any():
        first_valid_idx = valid[invalid_mask].long().argmax(dim=1)
        mask[invalid_mask, first_valid_idx] = True

    orig_cat = {k: v.clone() for k, v in cat_feats.items()}
    orig_cont = cont_feats.clone()
    masked_cat = {k: v.clone() for k, v in cat_feats.items()}
    masked_cont = cont_feats.clone()

    decide = torch.rand(B, S, device=device)
    zero_mask = mask & (decide < 0.8)
    rand_mask = mask & (decide >= 0.8) & (decide < 0.9)

    for col in masked_cat:
        masked_cat[col][zero_mask] = 0
        if rand_mask.any():
            card = cat_cards.get(col, 10) if cat_cards else 10
            masked_cat[col][rand_mask] = torch.randint(1, card, (rand_mask.sum(),), device=device)

    masked_cont[zero_mask] = 0.0
    if rand_mask.any():
        masked_cont[rand_mask] = torch.randn(rand_mask.sum(), masked_cont.shape[-1], device=device)

    return masked_cat, masked_cont, orig_cat, orig_cont, mask


D_MODEL = 128
NHEAD = 4
NUM_LAYERS = 3
DIM_FF = 256
DROPOUT = 0.1
EMB_DIM = 16
PRETRAIN_LR = 5e-4
PRETRAIN_EPOCHS = 20
PRETRAIN_WEIGHT_DECAY = 0.01
MASK_PROB = 0.15


model_pt = TransactionTransformer(
    cat_cardinalities=CAT_CARDS, cat_cols=CAT_COLS,
    d_model=D_MODEL, nhead=NHEAD, num_layers=NUM_LAYERS,
    dim_feedforward=DIM_FF, dropout=DROPOUT, num_targets=len(TARGET_COLS),
    emb_dim=EMB_DIM, n_cont_features=len(CONT_COLS),
).to(DEVICE)

n_params = sum(p.numel() for p in model_pt.parameters())
n_encoder = sum(p.numel() for n, p in model_pt.named_parameters() if n.startswith(ENCODER_PREFIXES))

print(f"Model parameters: {n_params:,} total | {n_encoder:,} encoder")

pt_optimizer = torch.optim.AdamW(
    model_pt.parameters(), lr=PRETRAIN_LR, weight_decay=PRETRAIN_WEIGHT_DECAY
)
pt_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    pt_optimizer, T_max=PRETRAIN_EPOCHS, eta_min=1e-6,
)


def pretrain_epoch(model, loader, optimizer, mask_prob=0.15, device=DEVICE):
    model.train()

    metrics = {"loss": 0.0, "cat_loss": 0.0, "cont_loss": 0.0}
    n_batches = 0

    for cat_feats, cont_feats, time_deltas, pad_mask in loader:
        cat_feats = {k: v.to(device) for k, v in cat_feats.items()}
        cont_feats = cont_feats.to(device)
        time_deltas = time_deltas.to(device)
        pad_mask = pad_mask.to(device)

        masked_cat, masked_cont, orig_cat, orig_cont, mask = mask_transactions(
            cat_feats, cont_feats, pad_mask, mask_prob, CAT_CARDS
        )

        if not mask.any():
            continue

        cat_logits, cont_pred = model.forward_pretrain(
            masked_cat, masked_cont, time_deltas, pad_mask, mask
        )

        cat_loss = sum(
            F.cross_entropy(cat_logits[col], orig_cat[col][mask])
            for col in cat_logits
        ) / len(cat_logits)

        cont_loss = F.mse_loss(cont_pred, orig_cont[mask])
        loss = cat_loss + cont_loss

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        metrics["loss"] += loss.item()
        metrics["cat_loss"] += cat_loss.item()
        metrics["cont_loss"] += cont_loss.item()
        n_batches += 1

    return tuple(val / max(n_batches, 1) for val in metrics.values())

print("Pre-training setup ready.")

In [ ]:
import pickle
from pathlib import Path

save_dir = Path(SAVE_DIR)
pt_history = {"loss": [], "cat_loss": [], "cont_loss": []}

print(f"Starting MTM pre-training ({PRETRAIN_EPOCHS} epochs)...")
print(f"{'Epoch':>6} | {'Loss':>8} | {'Cat CE':>8} | {'Cont MSE':>8} | {'LR':>8}")
print("-" * 53)

for epoch in range(1, PRETRAIN_EPOCHS + 1):
    loss, cat_loss, cont_loss = pretrain_epoch(
        model_pt, pretrain_loader, pt_optimizer, MASK_PROB
    )
    pt_scheduler.step()

    pt_history["loss"].append(loss)
    pt_history["cat_loss"].append(cat_loss)
    pt_history["cont_loss"].append(cont_loss)

    lr_now = pt_scheduler.get_last_lr()[0]
    print(f"{epoch:6d} | {loss:8.4f} | {cat_loss:8.4f} | {cont_loss:8.4f} | {lr_now:8.2e}")

pretrained_encoder_state = get_encoder_state(model_pt)
print(f"\nPre-training complete. Encoder state extracted ({len(pretrained_encoder_state)} tensors).")

torch.save(pretrained_encoder_state, save_dir / "pretrained_encoder.pt")
torch.save(model_pt.state_dict(), save_dir / "model_pretrain_full.pt")

with open(save_dir / "pt_history.pkl", "wb") as f:
    pickle.dump(pt_history, f)

print(f"Artifacts successfully saved to: {save_dir.as_posix()}/")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

plots_config = [
    (pt_history["loss"], "#4C72B0", "MTM Pre-training: Total Loss", "Total Loss"),
    (pt_history["cat_loss"], "#DD8452", "Categorical Reconstruction", "Cross-Entropy"),
    (pt_history["cont_loss"], "#55A868", "Amount Reconstruction", "MSE")
]

for ax, (data, color, title, ylabel) in zip(axes, plots_config):
    ax.plot(data, color=color, linewidth=2)

    ax.set(title=title, xlabel="Epoch", ylabel=ylabel)

    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", linestyle="--", alpha=0.6)

fig.suptitle("Masked Transaction Modeling — Pre-training Losses", fontsize=14, fontweight="bold", y=1.05)
plt.tight_layout()
plt.show()

In [ ]:
import torch
import os

D_MODEL = 128
NHEAD = 4
NUM_LAYERS = 3
DIM_FF = 256
DROPOUT = 0.1
EMB_DIM = 16


print(f"Configuração do Modelo: d={D_MODEL}, heads={NHEAD}, layers={NUM_LAYERS}")

encoder_path = os.path.join(SAVE_DIR, "pretrained_encoder.pt")

if os.path.exists(encoder_path):
    print(f"Carregando encoder pré-treinado de: {encoder_path}")
    pretrained_encoder_state = torch.load(encoder_path, map_location=DEVICE, weights_only=True)
    print(f"Sucesso! Encoder state carregado ({len(pretrained_encoder_state)} tensores).")
else:
    raise FileNotFoundError(f"Arquivo não encontrado: {encoder_path}.")

## 8. Classificação Supervisionada

Comparação direta de duas abordagens:

| | **Fine-Tuned** | **Do Zero (Scratch)** |
|---|---|---|
| Inicialização | Encoder pré-treinado (MTM) | Pesos aleatórios |
| LR | 1e-4 | 3e-4 (padrão) |
| Hipótese | Representações pré-treinadas aceleram convergência e melhoram generalização | Baseline para medir o ganho do pré-treinamento |

In [ ]:
import copy
from sklearn.metrics import roc_auc_score

class SigmoidFocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0, label_smoothing=0.02):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.ls = label_smoothing

    def forward(self, logits, targets):
        if self.ls > 0:
            targets = targets * (1 - self.ls) + 0.5 * self.ls

        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")

        probs = torch.sigmoid(logits)
        p_t = probs * targets + (1 - probs) * (1 - targets)

        focal_weight = (1 - p_t) ** self.gamma
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)

        return (alpha_t * focal_weight * bce).mean()


# Hiperparâmetros
FINETUNE_HEAD_LR = 1e-4
FINETUNE_ENCODER_LR = 5e-6
SCRATCH_LR = 3e-4
FINETUNE_EPOCHS = 20
PATIENCE = 5
WARMUP_EPOCHS = 3
WEIGHT_DECAY = 0.1
ACCUM_STEPS = 4
FOCAL_ALPHA = 0.75
FOCAL_GAMMA = 2.0
LABEL_SMOOTHING = 0.05
FT_DROPOUT = 0.3


def run_epoch(model, loader, criterion, optimizer=None, accum_steps=1, device=DEVICE):
    is_train = optimizer is not None
    model.train(is_train)

    total_loss = 0.0
    all_preds, all_targets = [], []

    if is_train:
        optimizer.zero_grad()

    with torch.set_grad_enabled(is_train):
        for batch_idx, (cat_feats, cont_feats, time_deltas, pad_mask, targets) in enumerate(loader):
            cat_feats = {k: v.to(device, non_blocking=True) for k, v in cat_feats.items()}
            cont_feats = cont_feats.to(device, non_blocking=True)
            time_deltas = time_deltas.to(device, non_blocking=True)
            pad_mask = pad_mask.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)

            logits = model(cat_feats, cont_feats, time_deltas, pad_mask)
            loss = criterion(logits, targets)

            if is_train:
                (loss / accum_steps).backward()

                if (batch_idx + 1) % accum_steps == 0 or (batch_idx + 1) == len(loader):
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()
                    optimizer.zero_grad()

            total_loss += loss.item()
            all_preds.append(torch.sigmoid(logits).detach().cpu().numpy())
            all_targets.append(targets.cpu().numpy())

    all_preds = np.concatenate(all_preds)
    all_targets = np.concatenate(all_targets)

    aucs = []
    for i in range(all_targets.shape[1]):
        try:
            aucs.append(roc_auc_score(all_targets[:, i], all_preds[:, i]))
        except ValueError:
            aucs.append(0.5)

    return total_loss / len(loader), np.mean(aucs), aucs


def build_param_groups(model, encoder_lr, head_lr, weight_decay):
    """Discriminative LR: lower for pre-trained encoder, higher for classifier head."""
    encoder_params, head_params = [], []
    for name, param in model.named_parameters():
        if name.startswith(ENCODER_PREFIXES):
            encoder_params.append(param)
        else:
            head_params.append(param)

    return [
        {"params": encoder_params, "lr": encoder_lr, "weight_decay": weight_decay},
        {"params": head_params, "lr": head_lr, "weight_decay": weight_decay},
    ]


def train_classifier(model, train_loader, val_loader, test_loader, criterion,
                     lr, num_epochs, patience, warmup_epochs, accum_steps,
                     weight_decay, model_name="Model", encoder_lr=None):

    if encoder_lr is not None:
        param_groups = build_param_groups(model, encoder_lr, lr, weight_decay)
        optimizer = torch.optim.AdamW(param_groups)
        print(f"Discriminative LR -> Encoder: {encoder_lr:.1e} | Head: {lr:.1e}")
    else:
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    warmup = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, total_iters=warmup_epochs)
    cosine = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs - warmup_epochs, eta_min=1e-6)
    scheduler = torch.optim.lr_scheduler.SequentialLR(optimizer, [warmup, cosine], milestones=[warmup_epochs])

    history = {"train_loss": [], "val_loss": [], "train_auc": [], "val_auc": []}
    best_val_auc = 0.0
    best_epoch = 0
    patience_counter = 0
    best_state = None

    print(f"\n{'='*65}\nTraining: {model_name} (Epochs: {num_epochs})\n{'='*65}")
    print(f"{'Epoch':>6} | {'Train Loss':>10} | {'Val Loss':>10} | {'Train AUC':>10} | {'Val AUC':>10}")
    print("-" * 65)

    for epoch in range(1, num_epochs + 1):
        train_loss, train_auc, _ = run_epoch(model, train_loader, criterion, optimizer, accum_steps)
        val_loss, val_auc, _ = run_epoch(model, val_loader, criterion)
        scheduler.step()

        for k, v in zip(history.keys(), [train_loss, val_loss, train_auc, val_auc]):
            history[k].append(v)

        marker = " 🌟" if val_auc > best_val_auc else ""
        print(f"{epoch:6d} | {train_loss:10.4f} | {val_loss:10.4f} | {train_auc:10.4f} | {val_auc:10.4f}{marker}")

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_epoch = epoch
            patience_counter = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"\nEarly stopping triggered. Best: Epoch {best_epoch} (Val AUC: {best_val_auc:.4f})")
                break

    if best_state:
        model.load_state_dict(best_state)
        model = model.to(DEVICE)

    print(f"\n[Test Evaluation] Restored best model from Epoch {best_epoch}")
    _, test_auc, test_aucs = run_epoch(model, test_loader, criterion)

    print(f"Mean Test AUC: {test_auc:.4f}")
    for col, auc in zip(TARGET_COLS, test_aucs):
        print(f"  {col}: {auc:.4f}")

    return model, history, best_val_auc, test_auc, test_aucs

In [ ]:
print("--- Initializing Fine-Tuning Pipeline ---")

criterion = SigmoidFocalLoss(
    alpha=FOCAL_ALPHA,
    gamma=FOCAL_GAMMA,
    label_smoothing=LABEL_SMOOTHING
)

model_ft = TransactionTransformer(
    cat_cardinalities=CAT_CARDS,
    cat_cols=CAT_COLS,
    d_model=D_MODEL,
    nhead=NHEAD,
    num_layers=NUM_LAYERS,
    dim_feedforward=DIM_FF,
    dropout=FT_DROPOUT,
    num_targets=len(TARGET_COLS),
    emb_dim=EMB_DIM,
    n_cont_features=len(CONT_COLS)
).to(DEVICE)

model_ft = load_encoder_state(model_ft, pretrained_encoder_state)

print(
    f"Pre-trained encoder weights loaded successfully.\n"
    f"Regularization -> Dropout: {FT_DROPOUT} | Weight Decay: {WEIGHT_DECAY} | Label Smoothing: {LABEL_SMOOTHING}"
)

# Execução do pipeline
model_ft, hist_ft, best_val_ft, test_auc_ft, test_aucs_ft = train_classifier(
    model=model_ft,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    criterion=criterion,
    lr=FINETUNE_HEAD_LR,
    encoder_lr=FINETUNE_ENCODER_LR,
    num_epochs=FINETUNE_EPOCHS,
    patience=PATIENCE,
    warmup_epochs=WARMUP_EPOCHS,
    accum_steps=ACCUM_STEPS,
    weight_decay=WEIGHT_DECAY,
    model_name="Transformer (Fine-Tuned)"
)

In [ ]:
import pickle
from pathlib import Path

save_dir = Path(SAVE_DIR)

torch.save(model_ft.state_dict(), save_dir / "model_finetuned.pt")

training_summary = {
    "finetune": {
        "best_val_auc": best_val_ft,
        "test_auc": test_auc_ft,
        "test_aucs": test_aucs_ft
    }
}

artifacts = {
    "hist_finetune.pkl": hist_ft,
    "training_summary.pkl": training_summary
}

for filename, data in artifacts.items():
    with open(save_dir / filename, "wb") as f:
        pickle.dump(data, f)

print(
    f"Artifacts successfully saved to: {save_dir.as_posix()}/\n"
    f"  - model_finetuned.pt\n"
    f"  - hist_finetune.pkl\n"
    f"  - training_summary.pkl"
)

In [ ]:
import os
import torch

print("Configurando variáveis e redefinindo a arquitetura correta...")

model_ft = TransactionTransformer(
    cat_cardinalities=CAT_CARDS,
    cat_cols=CAT_COLS,
    d_model=D_MODEL,
    nhead=NHEAD,
    num_layers=NUM_LAYERS,
    dim_feedforward=DIM_FF,
    dropout=FT_DROPOUT,
    num_targets=len(TARGET_COLS),
    emb_dim=EMB_DIM,
    n_cont_features=len(CONT_COLS)
).to(DEVICE)

finetuned_path = os.path.join(SAVE_DIR, "model_finetuned.pt")

if os.path.exists(finetuned_path):
    state_dict = torch.load(finetuned_path, map_location=DEVICE, weights_only=True)
    model_ft.load_state_dict(state_dict)
    model_ft.eval()
    print("Sucesso! model_ft (Fine-Tuned) carregado com os parâmetros corretos e pronto para uso.")
else:
    raise FileNotFoundError(f"Não encontrei o arquivo em: {finetuned_path}")

In [ ]:
print("--- Initializing Training from Scratch (Baseline) ---")
print("Random initialization, NO pre-training.")

criterion = SigmoidFocalLoss(
    alpha=FOCAL_ALPHA,
    gamma=FOCAL_GAMMA,
    label_smoothing=LABEL_SMOOTHING
)

model_scratch = TransactionTransformer(
    cat_cardinalities=CAT_CARDS,
    cat_cols=CAT_COLS,
    d_model=D_MODEL,
    nhead=NHEAD,
    num_layers=NUM_LAYERS,
    dim_feedforward=DIM_FF,
    dropout=DROPOUT,
    num_targets=len(TARGET_COLS),
    emb_dim=EMB_DIM,
    n_cont_features=len(CONT_COLS)
).to(DEVICE)

# Execução do baseline
model_scratch, hist_sc, best_val_sc, test_auc_sc, test_aucs_sc = train_classifier(
    model=model_scratch,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    criterion=criterion,
    lr=SCRATCH_LR,
    encoder_lr=None,
    num_epochs=20,
    patience=PATIENCE,
    warmup_epochs=WARMUP_EPOCHS,
    accum_steps=ACCUM_STEPS,
    weight_decay=WEIGHT_DECAY,
    model_name="Transformer (Scratch Baseline)"
)

In [ ]:
import pickle
from pathlib import Path

save_dir = Path(SAVE_DIR)
summary_path = save_dir / "training_summary.pkl"

torch.save(model_scratch.state_dict(), save_dir / "model_scratch.pt")

if summary_path.exists():
    with open(summary_path, "rb") as f:
        training_summary = pickle.load(f)
else:
    training_summary = {}

training_summary["scratch"] = {
    "best_val_auc": best_val_sc,
    "test_auc": test_auc_sc,
    "test_aucs": test_aucs_sc
}

artifacts = {
    "hist_scratch.pkl": hist_sc,
    "training_summary.pkl": training_summary
}

for filename, data in artifacts.items():
    with open(save_dir / filename, "wb") as f:
        pickle.dump(data, f)

print(
    f"Artifacts successfully saved to: {save_dir.as_posix()}/\n"
    f"  - model_scratch.pt\n"
    f"  - hist_scratch.pkl\n"
    f"  - training_summary.pkl (Updated with both models)"
)

In [ ]:
import torch
from pathlib import Path

print("Configuring architecture and loading scratch model weights...")

model_scratch = TransactionTransformer(
    cat_cardinalities=CAT_CARDS,
    cat_cols=CAT_COLS,
    d_model=D_MODEL,
    nhead=NHEAD,
    num_layers=NUM_LAYERS,
    dim_feedforward=DIM_FF,
    dropout=DROPOUT,
    num_targets=len(TARGET_COLS),
    emb_dim=EMB_DIM,
    n_cont_features=len(CONT_COLS)
).to(DEVICE)

scratch_path = Path(SAVE_DIR) / "model_scratch.pt"

if scratch_path.exists():
    state_dict = torch.load(scratch_path, map_location=DEVICE, weights_only=True)
    model_scratch.load_state_dict(state_dict)
    model_scratch.eval()
    print(f"Success! Model loaded and set to eval mode from: {scratch_path.name}")
else:
    raise FileNotFoundError(f"Weight file not found at: {scratch_path}")

In [ ]:
import pickle
import matplotlib.pyplot as plt
from pathlib import Path

save_dir = Path(SAVE_DIR)

files_to_load = {
    "hist_ft": "hist_finetune.pkl",
    "hist_sc": "hist_scratch.pkl",
    "summary": "training_summary.pkl"
}

loaded_data = {}
for key, filename in files_to_load.items():
    file_path = save_dir / filename
    if file_path.exists():
        with open(file_path, "rb") as f:
            loaded_data[key] = pickle.load(f)
        print(f"Success: {filename} loaded.")
    else:
        loaded_data[key] = {}
        print(f"Warning: {filename} not found.")

hist_ft = loaded_data["hist_ft"]
hist_sc = loaded_data["hist_sc"]
summary = loaded_data["summary"]

best_val_ft = summary.get("finetune", {}).get("best_val_auc", 0.0)
best_val_sc = summary.get("scratch", {}).get("best_val_auc", 0.0)


fig, axes = plt.subplots(1, 2, figsize=(16, 6))

plots_config = [
    ("auc", "ROC-AUC", "Fine-Tuned vs Scratch: ROC-AUC"),
    ("loss", "Focal Loss", "Fine-Tuned vs Scratch: Loss")
]

for ax, (metric, ylabel, title) in zip(axes, plots_config):
    if hist_ft:
        ax.plot(hist_ft[f"val_{metric}"], label="Fine-Tuned (Val)", color="#2ca02c", linewidth=2)
        ax.plot(hist_ft[f"train_{metric}"], label="Fine-Tuned (Train)", color="#2ca02c", alpha=0.3, linestyle="--")

    if hist_sc:
        ax.plot(hist_sc[f"val_{metric}"], label="Scratch (Val)", color="#d62728", linewidth=2)
        ax.plot(hist_sc[f"train_{metric}"], label="Scratch (Train)", color="#d62728", alpha=0.3, linestyle="--")

    ax.set(xlabel="Epoch", ylabel=ylabel, title=title)
    ax.legend()
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", linestyle="--", alpha=0.6)

    if metric == "auc":
        ax.axhline(y=0.5, color="gray", linestyle=":", alpha=0.5)

fig.suptitle("Ablation: Pre-training Effect on Supervised Classification", fontsize=14, fontweight="bold", y=1.05)
plt.tight_layout()
plt.show()

if best_val_ft and best_val_sc:
    gain = best_val_ft - best_val_sc
    print(
        f"\n[Ablation Result] Pre-training gain:\n"
        f"  Val AUC {best_val_ft:.4f} (Fine-Tuned) vs {best_val_sc:.4f} (Scratch) = +{gain:.4f}"
    )

## 9. Resultados: Baselines + Transformers

In [ ]:
summary_finetuning_path = os.path.join(SAVE_DIR, "training_summary_ft.pkl")
with open(summary_finetuning_path, "rb") as f:
  summary_finetuning = pickle.load(f)

summary_scratch_path = os.path.join(SAVE_DIR, "training_summary_sc.pkl")
with open(summary_scratch_path, "rb") as f:
  summary_scratch = pickle.load(f)

In [ ]:
test_auc_ft = summary_finetuning["finetune"]["test_auc"]
test_aucs_ft = summary_finetuning["finetune"]["test_aucs"]

test_auc_sc = summary_scratch["scratch"]["test_auc"]
test_aucs_sc = summary_scratch["scratch"]["test_aucs"]

In [ ]:
all_results["Transformer (Fine-Tuned)"] = {
    TARGET_COLS[i]: test_aucs_ft[i] for i in range(len(TARGET_COLS))
}
all_results["Transformer (Fine-Tuned)"]["mean_auc"] = test_auc_ft

all_results["Transformer (Scratch)"] = {
    TARGET_COLS[i]: test_aucs_sc[i] for i in range(len(TARGET_COLS))
}
all_results["Transformer (Scratch)"]["mean_auc"] = test_auc_sc

results_df = pd.DataFrame(all_results).T
results_df.columns = TARGET_COLS + ["Mean AUC"]
results_df = results_df.round(4)
results_df = results_df.sort_values("Mean AUC", ascending=False)

print("=" * 70)
print("BASELINES + TRANSFORMERS COMPARISON")
print("=" * 70)
results_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

results_df[TARGET_COLS].plot(kind="bar", ax=axes[0], rot=25, zorder=3)
axes[0].set(title="ROC-AUC per Target", ylabel="ROC-AUC", ylim=(0.4, 1.0))
axes[0].legend(title="Target", bbox_to_anchor=(1.02, 1), loc="upper left")
axes[0].axhline(y=0.5, color="gray", linestyle="--", alpha=0.5, zorder=1)

mean_aucs = results_df["Mean AUC"].sort_values(ascending=True)

colors = [
    "#2ca02c" if "Fine-Tuned" in name
    else "#d62728" if "Scratch" in name
    else "#DD8452"
    for name in mean_aucs.index
]

axes[1].barh(mean_aucs.index, mean_aucs.values, color=colors, zorder=3)
axes[1].set(title="Mean ROC-AUC Comparison", xlabel="Mean ROC-AUC", xlim=(0.4, 1.0))

for i, val in enumerate(mean_aucs.values):
    axes[1].text(val + 0.005, i, f"{val:.4f}", va="center", fontsize=10)

for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)

axes[0].grid(axis="y", linestyle="--", alpha=0.5, zorder=0)
axes[1].grid(axis="x", linestyle="--", alpha=0.5, zorder=0)

plt.tight_layout()
plt.show()

## 10. Abordagem Híbrida: Embeddings Pré-treinados + Features Agregadas

O modelo fine-tuned produz embeddings ricos via CLS token. Combinamos esses embeddings com as features agregadas tradicionais e treinamos LR e LightGBM como classificadores finais.

Pipeline: **Extração → StandardScaler → PCA(32) → Concatenação com features → Classificação**

In [ ]:
from sklearn.pipeline import make_pipeline

def extract_embeddings(model, loader, device=DEVICE):
    model.eval()
    all_embs, all_targets = [], []

    with torch.inference_mode():
        for batch in tqdm(loader, desc="Extracting embeddings"):
            if len(batch) == 5:
                cat_feats, cont_feats, time_deltas, pad_mask, targets = batch
                all_targets.append(targets.numpy())
            else:
                cat_feats, cont_feats, time_deltas, pad_mask = batch

            cat_feats = {k: v.to(device, non_blocking=True) for k, v in cat_feats.items()}

            cls_emb = model(
                cat_feats,
                cont_feats.to(device, non_blocking=True),
                time_deltas.to(device, non_blocking=True),
                pad_mask.to(device, non_blocking=True),
                return_embedding=True
            )
            all_embs.append(cls_emb.cpu().numpy())

    embs = np.concatenate(all_embs)
    targets = np.concatenate(all_targets) if all_targets else None

    return embs, targets

loader_kwargs = {"batch_size": 256, "shuffle": False, "collate_fn": collate_fn, "num_workers": 0}

emb_loader_tr = DataLoader(train_dataset, **loader_kwargs)
emb_loader_vl = DataLoader(val_dataset, **loader_kwargs)
emb_loader_ts = DataLoader(test_dataset, **loader_kwargs)


print("Extracting embeddings from Fine-Tuned model...")
emb_train_raw, y_emb_train = extract_embeddings(model_ft, emb_loader_tr)
emb_val_raw, y_emb_val     = extract_embeddings(model_ft, emb_loader_vl)
emb_test_raw, y_emb_test   = extract_embeddings(model_ft, emb_loader_ts)

print(
    f"\nShapes -> Train: {emb_train_raw.shape} | Val: {emb_val_raw.shape} | Test: {emb_test_raw.shape}\n"
    f"Raw Stats -> Mean: {emb_train_raw.mean():.3f} | Std: {emb_train_raw.std():.3f}"
)


N_PCA = 32

emb_pipeline = make_pipeline(
    StandardScaler(),
    PCA(n_components=N_PCA, random_state=SEED)
)

print("\nFitting Scaler and PCA...")
emb_train = emb_pipeline.fit_transform(emb_train_raw)
emb_val   = emb_pipeline.transform(emb_val_raw)
emb_test  = emb_pipeline.transform(emb_test_raw)

pca_step = emb_pipeline.named_steps['pca']
print(f"After PCA({N_PCA}): Explained Variance = {pca_step.explained_variance_ratio_.sum():.4f}")

In [ ]:
pca_cols = [f"pca_{i}" for i in range(N_PCA)]
feat_scaler = StandardScaler()

def build_hybrid_sw(dataset, embeddings, df_features_by_month, df_features_global, pca_cols):
    """Build hybrid features merging manual aggregations and PCA embeddings."""
    feat_rows, valid_indices, cids, targets = [], [], [], []

    for i, ex in enumerate(dataset.examples):
        cid = ex.get("client_id")
        t_mon = ex.get("target_month")

        feat_src = df_features_by_month.get(t_mon, df_features_global)

        if cid in feat_src.index:
            feat_rows.append(feat_src.loc[cid])
            valid_indices.append(i)
            cids.append(cid)
            targets.append(ex["targets"])

    if not feat_rows:
        raise ValueError("No matching clients between dataset and features")

    feat_df = pd.DataFrame(feat_rows).reset_index(drop=True)

    emb_valid = pd.DataFrame(embeddings[valid_indices], columns=pca_cols)

    hyb_df = pd.concat([feat_df, emb_valid], axis=1)

    return feat_df, emb_valid, hyb_df, cids, np.array(targets)


print("Building hybrid datasets (Aggregated + PCA Embeddings)...")
feat_tr, emb_tr, hyb_tr, cids_tr, y_tr = build_hybrid_sw(
    train_dataset, emb_train, df_features_by_month, df_features, pca_cols
)
feat_vl, emb_vl, hyb_vl, cids_vl, y_vl = build_hybrid_sw(
    val_dataset, emb_val, df_features_by_month, df_features, pca_cols
)
feat_ts, emb_ts, hyb_ts, cids_ts, y_ts = build_hybrid_sw(
    test_dataset, emb_test, df_features_by_month, df_features, pca_cols
)

feat_tr_s = pd.DataFrame(feat_scaler.fit_transform(feat_tr), columns=feat_tr.columns)
feat_vl_s = pd.DataFrame(feat_scaler.transform(feat_vl), columns=feat_vl.columns)
feat_ts_s = pd.DataFrame(feat_scaler.transform(feat_ts), columns=feat_ts.columns)

hyb_tr_s = pd.concat([feat_tr_s, emb_tr], axis=1)
hyb_vl_s = pd.concat([feat_vl_s, emb_vl], axis=1)
hyb_ts_s = pd.concat([feat_ts_s, emb_ts], axis=1)

print(
    f"\nHybrid features ready: {hyb_tr_s.shape[1]} dims ({feat_tr.shape[1]} manual + {N_PCA} PCA)\n"
    f"Shapes -> Train: {len(hyb_tr_s)} | Val: {len(hyb_vl_s)} | Test: {len(hyb_ts_s)}"
)

In [ ]:
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import numpy as np

LGBM_PARAMS = {
    "n_estimators": 1000, "max_depth": 4, "learning_rate": 0.05,
    "num_leaves": 15, "min_child_samples": 100, "subsample": 0.7,
    "colsample_bytree": 0.7, "reg_alpha": 1.0, "reg_lambda": 5.0,
    "random_state": SEED, "verbose": -1, "n_jobs": -1,
}

configs = {
    "Hybrid (LR)":   {"train": hyb_tr_s, "test": hyb_ts_s, "cls": "lr"},
    "Hybrid (LGBM)": {"train": hyb_tr_s, "test": hyb_ts_s, "val": hyb_vl_s, "cls": "lgbm"},
    "Emb Only (LR)": {"train": emb_tr,   "test": emb_ts,   "cls": "lr"},
}

hybrid_results = {}

targets_header = " | ".join(f"{t[:6]:>6}" for t in TARGET_COLS)
print(f"{'Model':<15} | {targets_header} | Mean AUC")
print("-" * (30 + len(targets_header)))

for name, cfg in configs.items():
    X_tr = np.asarray(cfg["train"])
    X_ts = np.asarray(cfg["test"])

    aucs = []

    for i, target in enumerate(TARGET_COLS):
        y_train_i = y_tr[:, i]
        y_test_i = y_ts[:, i]

        if cfg["cls"] == "lr":
            mdl = LogisticRegression(
                max_iter=1000, C=1.0, class_weight="balanced",
                solver="lbfgs", random_state=SEED
            )
            mdl.fit(X_tr, y_train_i)

        elif cfg["cls"] == "lgbm":
            pos = y_train_i.sum()
            spw = min((len(y_train_i) - pos) / max(pos, 1), 100.0)

            X_vl = np.asarray(cfg["val"])
            y_val_i = y_vl[:, i]

            mdl = LGBMClassifier(**LGBM_PARAMS, scale_pos_weight=spw)
            mdl.fit(
                X_tr, y_train_i,
                eval_set=[(X_vl, y_val_i)],
                callbacks=[early_stopping(50, verbose=False), log_evaluation(0)]
            )

        y_pred = mdl.predict_proba(X_ts)[:, 1]
        aucs.append(roc_auc_score(y_test_i, y_pred))

    mean_auc = np.mean(aucs)

    hybrid_results[name] = {col: auc for col, auc in zip(TARGET_COLS, aucs)}
    hybrid_results[name]["mean_auc"] = mean_auc

    auc_str = " | ".join(f"{a:6.4f}" for a in aucs)
    print(f"{name:<15} | {auc_str} | {mean_auc:.4f}")

## 11. Comparação Final — Todos os Modelos

In [ ]:
all_final = {}
all_final.update(all_results)
all_final.update(hybrid_results)

final_df = pd.DataFrame(all_final).T
final_df.columns = TARGET_COLS + ["Mean AUC"]
final_df = final_df.round(4).sort_values("Mean AUC", ascending=False)

print("=" * 75)
print("FINAL COMPARISON — ALL MODELS")
print("=" * 75)
final_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

final_df[TARGET_COLS].plot(kind="bar", ax=axes[0], rot=30, zorder=3)
axes[0].set(title="ROC-AUC per Target — All Models", ylabel="ROC-AUC", ylim=(0.4, 1.0))
axes[0].legend(title="Target", bbox_to_anchor=(1.02, 1), loc="upper left")
axes[0].axhline(y=0.5, color="gray", linestyle="--", alpha=0.5, zorder=1)

mean_aucs = final_df["Mean AUC"].sort_values(ascending=True)

def get_color(name):
    if "Fine-Tuned" in name: return "#2ca02c"
    if "Scratch" in name: return "#d62728"
    if "Hybrid" in name: return "#9467bd"
    if "Emb" in name: return "#8c564b"
    return "#DD8452"

bar_colors = [get_color(name) for name in mean_aucs.index]

axes[1].barh(mean_aucs.index, mean_aucs.values, color=bar_colors, zorder=3)
axes[1].set(title="Mean ROC-AUC — All Models", xlabel="Mean ROC-AUC", xlim=(0.4, 1.0))

for i, val in enumerate(mean_aucs.values):
    axes[1].text(val + 0.005, i, f"{val:.4f}", va="center", fontsize=10)

for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)

axes[0].grid(axis="y", linestyle="--", alpha=0.5, zorder=0)
axes[1].grid(axis="x", linestyle="--", alpha=0.5, zorder=0)

plt.tight_layout()
plt.show()


print("\n" + "=" * 65 + "\nKEY COMPARISONS\n" + "=" * 65)

pairs = [
    ("Fine-Tuned vs Scratch", "Transformer (Fine-Tuned)", "Transformer (Scratch)"),
    ("Hybrid (LR) vs LR baseline", "Hybrid (LR)", "Logistic Regression"),
    ("Fine-Tuned vs LR baseline", "Transformer (Fine-Tuned)", "Logistic Regression"),
]

for title, model_a, model_b in pairs:
    if model_a in final_df.index and model_b in final_df.index:
        print(f"\n[ {title} ]")

        for col in TARGET_COLS + ["Mean AUC"]:
            va = final_df.loc[model_a, col]
            vb = final_df.loc[model_b, col]
            delta = va - vb

            sign = "+" if delta >= 0 else ""
            print(f"  {col:<12}: {va:.4f} vs {vb:.4f} | Δ = {sign}{delta:.4f}")

### 11.1 Análise de Attention — Modelo Fine-Tuned

Visualização dos pesos de atenção do Transformer pré-treinado e fine-tuned.

In [ ]:
model_ft.eval()
sample_batch = next(iter(test_loader))
cat_feats, cont_feats, time_deltas, pad_mask, targets = sample_batch
cat_feats_dev = {k: v.to(DEVICE) for k, v in cat_feats.items()}

with torch.no_grad():
    logits, attn_weights = model_ft(
        cat_feats_dev, cont_feats.to(DEVICE), time_deltas.to(DEVICE),
        pad_mask.to(DEVICE), return_attention=True,
    )

cls_attn = attn_weights[-1][:, 0, 1:]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for i in range(min(5, len(cls_attn))):
    seq_len_i = (~pad_mask[i]).sum().item()
    attn_vals = cls_attn[i, :seq_len_i].cpu().numpy()
    axes[0].plot(attn_vals, alpha=0.6, label=f"Client {i}")

axes[0].set_xlabel("Transaction position")
axes[0].set_ylabel("CLS attention weight")
axes[0].set_title("[CLS] Attention — Last Layer (Fine-Tuned)")
axes[0].legend()

n_bins = 20
bin_edges = np.linspace(0, 1, n_bins + 1)
bin_means = np.zeros(n_bins)
bin_counts = np.zeros(n_bins)

for i in range(len(cls_attn)):
    seq_len_i = (~pad_mask[i]).sum().item()
    if seq_len_i > 0:
        attn_vals = cls_attn[i, :seq_len_i].cpu().numpy()
        positions = np.linspace(0, 1, len(attn_vals))
        for pos, att in zip(positions, attn_vals):
            bi = min(int(pos * n_bins), n_bins - 1)
            bin_means[bi] += att
            bin_counts[bi] += 1

bin_means /= np.maximum(bin_counts, 1)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

axes[1].bar(bin_centers, bin_means, width=1.0/n_bins*0.9, color="#2ca02c")
axes[1].set_xlabel("Normalized position (0=oldest, 1=newest)")
axes[1].set_ylabel("Average CLS attention")
axes[1].set_title("Where Does the Pre-Trained Transformer Focus?")

plt.tight_layout()
plt.show()

### 11.2 Feature Importance — Modelo Híbrido

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from sklearn.linear_model import LogisticRegression

if "Hybrid (LR)" in hybrid_results:
    n_targets = y_tr.shape[1]
    n_agg = feat_tr.shape[1]
    feat_names = hyb_tr_s.columns.tolist()

    plt.rcParams.update({'font.size': 10, 'axes.edgecolor': '#cccccc', 'axes.linewidth': 0.8})

    fig, axes = plt.subplots(n_targets, 2, figsize=(14, 4 * n_targets), dpi=300)

    if n_targets == 1:
        axes = np.expand_dims(axes, axis=0)

    for t_idx in range(n_targets):
        fi_model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
        fi_model.fit(hyb_tr_s.values, y_tr[:, t_idx])
        importances = np.abs(fi_model.coef_[0])

        sorted_idx = np.argsort(importances)[-15:]
        colors_fi = ["#4C72B0" if i < n_agg else "#55A868" for i in sorted_idx] # Cores mais sóbrias (Muted)

        ax_feat = axes[t_idx, 0]
        ax_feat.barh([feat_names[i] for i in sorted_idx], importances[sorted_idx], color=colors_fi, height=0.7)
        ax_feat.set_title(f"Panel {chr(65 + t_idx*2)}: Top 15 Features (Target {t_idx + 1})", fontsize=11, fontweight='bold', loc='left')
        ax_feat.set_xlabel("Relative Importance (|Coefficient|)")
        ax_feat.spines['top'].set_visible(False)
        ax_feat.spines['right'].set_visible(False)

        if t_idx == 0:
            ax_feat.legend(handles=[Patch(color="#4C72B0", label="Aggregated"),
                                    Patch(color="#55A868", label="Pre-trained Embedding (PCA)")],
                           loc="lower right", frameon=True, facecolor='white', edgecolor='none')

        imp_agg = importances[:n_agg].sum()
        imp_emb = importances[n_agg:].sum()
        total = imp_agg + imp_emb
        p_agg, p_emb = (imp_agg/total)*100, (imp_emb/total)*100

        ax_split = axes[t_idx, 1]
        ax_split.barh(["Composition"], [p_agg], color="#4C72B0", height=0.4, label="Aggregated" if t_idx==0 else "")
        ax_split.barh(["Composition"], [p_emb], left=[p_agg], color="#55A868", height=0.4, label="Embedding" if t_idx==0 else "")

        ax_split.set_title(f"Panel {chr(66 + t_idx*2)}: Importance Split (Target {t_idx + 1})", fontsize=11, fontweight='bold', loc='left')
        ax_split.set_xlabel("Percentage of Total Importance (%)")
        ax_split.set_xlim(0, 100)
        ax_split.spines['top'].set_visible(False)
        ax_split.spines['right'].set_visible(False)

        if total > 0:
            ax_split.text(p_agg/2, 0, f"{p_agg:.1f}%", ha="center", va="center", color="white", fontweight="bold", fontsize=11)
            ax_split.text(p_agg + (p_emb/2), 0, f"{p_emb:.1f}%", ha="center", va="center", color="white", fontweight="bold", fontsize=11)

    plt.tight_layout()
    plt.show()

In [ ]:
import pickle

final_df.to_csv(os.path.join(SAVE_DIR, "final_results.csv"))

with open(os.path.join(SAVE_DIR, "all_results.pkl"), "wb") as f:
    pickle.dump({"baselines": all_results, "hybrid": hybrid_results}, f)

with open(os.path.join(SAVE_DIR, "scalers.pkl"), "wb") as f:
    pickle.dump({"qt_amount": qt, "emb_scaler": emb_scaler, "pca": pca, "feat_scaler": feat_scaler}, f)

config = {
    "D_MODEL": D_MODEL, "NHEAD": NHEAD, "NUM_LAYERS": NUM_LAYERS,
    "DIM_FF": DIM_FF, "EMB_DIM": EMB_DIM, "N_CONT": N_CONT,
    "FT_DROPOUT": FT_DROPOUT, "WEIGHT_DECAY": WEIGHT_DECAY,
    "FINETUNE_HEAD_LR": FINETUNE_HEAD_LR, "FINETUNE_ENCODER_LR": FINETUNE_ENCODER_LR,
    "LABEL_SMOOTHING": LABEL_SMOOTHING, "FOCAL_ALPHA": FOCAL_ALPHA, "FOCAL_GAMMA": FOCAL_GAMMA,
    "PRETRAIN_EPOCHS": PRETRAIN_EPOCHS, "PRETRAIN_LR": PRETRAIN_LR,
    "BATCH_SIZE": BATCH_SIZE, "MAX_SEQ_LEN": MAX_SEQ_LEN,
    "CAT_COLS": CAT_COLS, "CONT_COLS": CONT_COLS,
}
with open(os.path.join(SAVE_DIR, "config.pkl"), "wb") as f:
    pickle.dump(config, f)

print(f"All artifacts saved to {SAVE_DIR}/")
print(f"Files:")
for fname in sorted(os.listdir(SAVE_DIR)):
    fsize = os.path.getsize(os.path.join(SAVE_DIR, fname))
    print(f"  {fname:40s} {fsize/1024:>8.1f} KB")